# ChemBreak V11 Cloud
## Standard Google Colab + Vertex AI

V11 is the quality-first task-bank pipeline agreed after reviewing V9 and V10.

Core design:
- Gemini 3.1 Pro generates all three controlled candidate slots.
- Python performs deterministic validation.
- Gemini 3.1 Pro repairs and refills.
- gpt-oss-120B and Gemini 2.5 Pro judge every active candidate set independently and concurrently.
- Gemini 3.1 Pro adjudicates genuine disagreements blindly.
- 3 valid candidates: judge all 3. 2 valid: judge both. 1 valid: targeted pre-judge refill, then dual single-candidate qualification if still alone. 0 valid: full refill.

A local GPU is not required because model inference runs remotely through Vertex AI. A CPU Colab runtime is sufficient.


## 1. Python setup


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os
import shutil

PROJECT_ID = "rs-foundsecft-mghasemi"

print("Python setup: OK")
print("Project:", PROJECT_ID)


## 2. Authenticate standard Colab to Google Cloud

Open the displayed URL, sign in with the Google account that has access to the project, then paste the verification code back into Colab.


In [ ]:
!gcloud auth application-default login --no-launch-browser


## 3. Attach the project to Application Default Credentials


In [ ]:
!gcloud auth application-default set-quota-project rs-foundsecft-mghasemi
!gcloud config set project rs-foundsecft-mghasemi

import google.auth

credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

print("Authentication: OK")
print("Detected project:", detected_project)
print("Quota project:", credentials.quota_project_id)
print("Credential type:", type(credentials).__name__)


## 4. Mount Google Drive for persistent checkpoints


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ChemBreak_V11")
else:
    STORAGE_ROOT = Path("/content/ChemBreak_V11")

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage root:", STORAGE_ROOT)


## 5. Clone or refresh the GitHub repository


In [ ]:
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/content/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V11_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in GitHub. "
        "Upload the complete V11 folder to the repository first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v11_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

print("V11 folder:", PROJECT_DIR)
print("Pipeline:", PIPELINE)


In [ ]:
# Hard verification gate: do not spend model calls if GitHub served the wrong build.
pipeline_source = PIPELINE.read_text(encoding="utf-8")
config_source = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))

checks = {
    "V11 version": 'VERSION = "11.0-cloud"' in pipeline_source,
    "fresh namespace": 'NAMESPACE = "CBV11C"' in pipeline_source,
    "visual progress bars": "def _bar(" in pipeline_source,
    "20-second heartbeat support": "heartbeat_seconds" in pipeline_source,
    "concurrent judges": "ThreadPoolExecutor" in pipeline_source,
    "pre-judge refill": "prejudge_refill_stage" in pipeline_source,
    "blind adjudication": "adjudicate_stage" in pipeline_source,
    "Gemini-only generator role": config_source.get("generator_role") == "generator",
    "exactly two judge roles": config_source.get("judge_roles") == ["judge_a", "judge_b"],
    "no Llama model role": not any("llama" in str(v).lower() for v in config_source.get("models", {}).values()),
}

for name, ok in checks.items():
    print(f"{name}: {'OK' if ok else 'MISSING'}")

if not all(checks.values()):
    raise RuntimeError(
        "GitHub is not serving the locked V11 build. Replace the ChemBreak_V11_Cloud "
        "folder with the current package, then rerun the clone/setup cells."
    )

print("ChemBreak V11 verification: PASSED")


## 6. Install V11 requirements


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)
print("V11 requirements installed.")


## 7. Choose the run

Keep `RUN_TYPE = "test"` until the complete 9-task workflow has been reviewed. The same pipeline then supports `pilot` (100 final tasks) and `production` (500 final tasks).


In [ ]:
RUN_TYPE = "test"  # "test", "pilot", or "production"
GCS_OUTPUT_URI = ""

RUNTIME_DIR = Path("/content/ChemBreak_V11_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

OUTPUT_DIR = STORAGE_ROOT / "outputs" / RUN_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    import os
    import subprocess
    import sys
    import time

    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(OUTPUT_DIR),
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    print(f"\n===== V11 {stage.upper()} START =====", flush=True)
    print(f"Pipeline: {PIPELINE}", flush=True)
    print(f"Output:   {OUTPUT_DIR}", flush=True)

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)

    process.stdout.close()
    return_code = process.wait()
    elapsed = time.time() - started

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

    print(
        f"===== V11 {stage.upper()} DONE | elapsed {elapsed/60:.1f} min =====\n",
        flush=True,
    )

print("Run type:", RUN_TYPE)
print("Output directory:", OUTPUT_DIR)


## 8. Preflight Vertex AI model access


In [ ]:
run_stage("preflight")

import pandas as pd
from IPython.display import display

preflight = pd.read_csv(OUTPUT_DIR / "preflight_models.csv")
display(preflight)

required_roles = {"generator", "repair_model", "judge_a", "judge_b", "adjudicator"}
bad = preflight[preflight["role"].isin(required_roles) & (preflight["status"] != "OK")]
if not bad.empty:
    raise RuntimeError(
        "V11 preflight failed for required roles:\n" +
        bad[["role", "model", "status", "detail"]].to_string(index=False)
    )

print("All required V11 model roles passed preflight.")


## 9. Bootstrap fresh source provenance


In [ ]:
run_stage("bootstrap")


## 10. Build the fresh V11 assignment plan


In [ ]:
run_stage("plan")


## 11. Inspect V11 coverage and candidate profiles before generation


In [ ]:
plan = pd.read_csv(OUTPUT_DIR / "assignments_v11.csv")
display(plan.head(25))
print("Assignments:", len(plan))

for label, column in [("HC", "hc_id"), ("HD", "hd_id"), ("OT", "ot_id")]:
    print(f"\n{label} coverage")
    display(plan[column].value_counts().sort_index())

profiles = pd.concat([
    plan["candidate_profile_a"],
    plan["candidate_profile_b"],
    plan["candidate_profile_c"],
])
print("\nCandidate-language profile coverage")
display(profiles.value_counts())


## 12. Generate 3 controlled Gemini 3.1 Pro candidates per assignment


In [ ]:
run_stage("generate")


## 13. Deterministic Python validation


In [ ]:
run_stage("validate")


## 14. Repair invalid initial candidates with Gemini 3.1 Pro


In [ ]:
run_stage("repair")


## 15. Restore competition when only one valid candidate survives

V11 makes up to two targeted pre-judge refill attempts. If only one candidate still survives, the two judges switch to single-candidate qualification rather than dropping the assignment.


In [ ]:
run_stage("prejudge_refill")


## 16. Two independent judges, concurrently

Judge A is gpt-oss-120B. Judge B is Gemini 2.5 Pro. Both evaluate every active candidate set independently. A parser or API failure is not a vote and will be retried on rerun.


In [ ]:
run_stage("judge")


## 17. Blind adjudication of genuine disagreements


In [ ]:
run_stage("adjudicate")


## 18. Recovery loop for unresolved assignments

This loop generates two fresh Gemini candidates for assignments with zero valid candidates or a final reject decision. Each refill is validated, invalid refill candidates are repaired, one-candidate sets receive pre-judge refill, then both judges run again. Maximum full refill cycles are controlled in `config/run_config.json`.


In [ ]:
cfg_now = json.loads(RUNTIME_CONFIG.read_text(encoding="utf-8"))
max_cycles = int(cfg_now["recovery"]["max_full_refill_cycles"])
target = (
    cfg_now["test_target"] if RUN_TYPE == "test" else
    cfg_now["pilot_target"] if RUN_TYPE == "pilot" else
    cfg_now["production_target"]
)

def current_selected_count():
    path = OUTPUT_DIR / "selected_tasks.csv"
    if not path.exists():
        return 0
    return len(pd.read_csv(path).drop_duplicates(subset=["assignment_id"]))

for recovery_cycle in range(1, max_cycles + 1):
    if current_selected_count() >= target:
        print(f"Target reached: {current_selected_count()}/{target}")
        break
    print(f"\n######## RECOVERY CYCLE {recovery_cycle}/{max_cycles} ########")
    run_stage("refill")
    run_stage("prejudge_refill")
    run_stage("judge")
    run_stage("adjudicate")
    run_stage("status")

print(f"Recovery loop finished with {current_selected_count()}/{target} selected assignments.")


## 19. Finalize and inspect the task bank


In [ ]:
run_stage("finalize")
run_stage("status")

for name in [
    "run_summary.json",
    "pipeline_metrics.json",
    "coverage_report.csv",
    "diversity_report.csv",
    "similarity_summary.json",
    "final_task_bank.csv",
]:
    path = OUTPUT_DIR / name
    print("\n", name)
    if path.suffix == ".json" and path.exists():
        print(path.read_text(encoding="utf-8"))
    elif path.exists():
        display(pd.read_csv(path).head(40))
    else:
        print("not written")


## 20. Create a persistent checkpoint ZIP


In [ ]:
summary = json.loads((OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8"))
label = summary["completion_label"]
archive = shutil.make_archive(
    str(STORAGE_ROOT / f"ChemBreak_V11_{RUN_TYPE}_{label}"),
    "zip",
    root_dir=str(OUTPUT_DIR),
)
print("Checkpoint ZIP:", archive)
